# Outliers with the IQR method
Detecting outliers in the skewed column `placement_exam_marks` with the box-plot fences, then trimming and capping them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split

## Load the data
1000 students: CGPA, marks in the placement exam (out of 100), and whether they were placed (1) or not (0).

In [ ]:
df = pd.read_csv("data/placement.csv")
print(df.shape)
df.head()

## Check the shape of each column
The IQR method is meant for a skewed column.

In [ ]:
# one histogram per column, side by side
fig = make_subplots(1, 2, subplot_titles=["cgpa", "placement_exam_marks"])
fig.add_trace(go.Histogram(x=df["cgpa"]), 1, 1)
fig.add_trace(go.Histogram(x=df["placement_exam_marks"]), 1, 2)
fig.update_layout(template="simple_white", showlegend=False, height=350)
fig.show()

In [ ]:
# skewness: near 0 = symmetric, above 0 = long right tail
df[["cgpa", "placement_exam_marks"]].skew()

In [ ]:
# the summary numbers of the skewed column (25%, 50%, 75% are Q1, median, Q3)
df["placement_exam_marks"].describe()

In [ ]:
# box plot: the dots beyond the right whisker are the outliers
# quartilemethod="linear" computes the quartiles the same way as pandas
go.Figure(go.Box(x=df["placement_exam_marks"], quartilemethod="linear", name="")).update_layout(
    template="simple_white", height=250)

## Q1, Q3, IQR and the fences

In [ ]:
# 25th and 75th percentiles
percentile25 = df["placement_exam_marks"].quantile(0.25)
percentile75 = df["placement_exam_marks"].quantile(0.75)
iqr = percentile75 - percentile25
print("Q1", percentile25, " Q3", percentile75, " IQR", iqr)

In [ ]:
# the fences: 1.5 IQR beyond each edge of the box
upper_limit = percentile75 + 1.5 * iqr
lower_limit = percentile25 - 1.5 * iqr
print("upper limit", upper_limit)
print("lower limit", lower_limit)

## Finding the outliers

In [ ]:
# above the upper fence
df[df["placement_exam_marks"] > upper_limit]

In [ ]:
# below the lower fence: none, since no mark is below 0
df[df["placement_exam_marks"] < lower_limit]

## Trimming

In [ ]:
# keep the rows inside the fences (both ends, to be safe)
new_df = df[df["placement_exam_marks"].between(lower_limit, upper_limit)]
new_df.shape

## Capping

In [ ]:
new_df_cap = df.copy()
# np.where(condition, value if true, value if false), nested twice:
# above the upper fence -> upper fence; below the lower fence -> lower fence; else unchanged
new_df_cap["placement_exam_marks"] = np.where(
    new_df_cap["placement_exam_marks"] > upper_limit,
    upper_limit,
    np.where(new_df_cap["placement_exam_marks"] < lower_limit,
             lower_limit,
             new_df_cap["placement_exam_marks"]),
)
print(new_df_cap.shape)
new_df_cap["placement_exam_marks"].describe()

In [ ]:
# the same thing in one line: clip pulls every value into [lower, upper]
same = df["placement_exam_marks"].clip(lower_limit, upper_limit)
(same == new_df_cap["placement_exam_marks"]).all()

## Comparing: original, trimmed, capped
Histogram on the left, box plot on the right.

In [ ]:
versions = {"original": df, "trimmed": new_df, "capped": new_df_cap}
fig = make_subplots(3, 2, shared_xaxes=True, subplot_titles=[t for k in versions for t in (k, "")])
for i, part in enumerate(versions.values(), start=1):
    marks = part["placement_exam_marks"]
    fig.add_trace(go.Histogram(x=marks, xbins=dict(start=0, end=104, size=4)), i, 1)
    fig.add_trace(go.Box(x=marks, quartilemethod="linear", name=""), i, 2)
fig.update_layout(template="simple_white", showlegend=False, height=700)
fig.show()

In [ ]:
# why the trimmed box plot shows a new dot: its fences are computed again on 985 rows
m = new_df["placement_exam_marks"]
q1, q3 = m.quantile([0.25, 0.75])
print("new upper fence", q3 + 1.5 * (q3 - q1), " largest mark left", m.max())

## Better practice: learn the fences on the training set
The fences are learned from data, like a scaler's mean. We split first, learn them on the training rows, and apply the same fences to the test rows.

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
# fences from the training set only
q1, q3 = train["placement_exam_marks"].quantile([0.25, 0.75])
lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
print("training fences", lo, hi)

In [ ]:
# outliers in each part, judged with the training fences
for name, part in [("train", train), ("test", test)]:
    marks = part["placement_exam_marks"]
    out = marks[(marks < lo) | (marks > hi)]
    print(name, len(part), "rows,", len(out), "outliers:", sorted(out.tolist()))

In [ ]:
# trim the training set only; cap both parts with the training fences
train_trimmed = train[train["placement_exam_marks"].between(lo, hi)]
train_capped = train.assign(placement_exam_marks=train["placement_exam_marks"].clip(lo, hi))
test_capped = test.assign(placement_exam_marks=test["placement_exam_marks"].clip(lo, hi))
print(train_trimmed.shape, train_capped["placement_exam_marks"].max(), test_capped["placement_exam_marks"].max())

In [ ]:
# what happens if we trim again and again, recomputing the fences each round
m = df["placement_exam_marks"]
for round_ in range(1, 11):
    q1, q3 = m.quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    out = (m > upper).sum()
    print("round", round_, " upper fence", upper, " outliers", out, " rows left", len(m) - out)
    if out == 0:
        break
    m = m[m <= upper]